# Modelos de Analítica (T8.4)

Retraso (GradientBoosting, D81) y anomalías de combustible (Isolation Forest, D09), entrenados con la semilla fija sobre `plataforma/datos-sinteticos` (T8.5). Es el mismo código que corre en el build de la imagen (`ml/entrenar.py`).

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from ml.entrenar import cargas_con_referencia, entrenar_anomalias, entrenar_retraso, muestras

DATOS = pathlib.Path.cwd().parents[2] / "plataforma" / "datos-sinteticos"
m = muestras(DATOS)
m.describe()

## Retraso: recall y matriz de confusión (meta: recall de *retrasado* ≥ 0,75, D82)

Etiqueta (D80): la duración real supera 1,25 veces la planificada. Validación con el 25 % de los viajes (todas las muestras de un viaje van al mismo lado).

In [ ]:
retraso, metricas = entrenar_retraso(m)
import pandas as pd

print("recall", metricas["recall"], "precision", metricas["precision"])
pd.DataFrame(
    metricas["matriz"], index=["real a tiempo", "real retrasado"], columns=["predicho a tiempo", "predicho retrasado"]
)

In [ ]:
importancias = retraso["modelo"].named_steps["gb"].feature_importances_
nombres = retraso["modelo"].named_steps["columnas"].get_feature_names_out()
pd.Series(importancias, index=nombres).sort_values(ascending=False)

## Anomalías de combustible

`IsolationForest(contamination=0.05, n_estimators=100, random_state=42)` sobre `[kmPorLitro, litros]`. FF2: 450 L con 120 km (0,27 km/L) es anomalía; 180 L con 450 km (2,5 km/L) no.

In [ ]:
anomalias, resumen = entrenar_anomalias(cargas_con_referencia(DATOS))
print(resumen)
x = pd.DataFrame([[0.267, 450], [2.5, 180]], columns=["kmPorLitro", "litros"])
anomalias["modelo"].predict(x)